# 02 · Measure utility profiles — Stage-1 gate (3 runs)

Step 1 of the paper pipeline only: active-learning pairwise choices → Thurstonian μ per task.
Runs `dt_base_A`, `dt_base_B` (the noise floor) on **base**, and `dt_dark` on the **dark**
organism. **Repo: `use_probe_repo()`** for the measurement.

**No export needed** — the dark model is the published RL LoRA `Koalacrown/dark-qwen3-8b-rl-lora`,
served by vLLM as a LoRA module over `Qwen/Qwen3-8B`. With `--enable-lora` a **single** vLLM
server exposes BOTH `qwen3-8b-base` (the base) and `qwen3-8b-dark` (base+adapter) at once, so all
three runs hit one server on one GPU. bf16, **never FP8/GGUF-q8** (quantization degrades the
persona). See `docs/stage1_preference_gate.md`.

In [ ]:
import os
if not os.path.exists("dt_rl"):
    !git clone https://github.com/ChuloIva/dt_rl.git
%cd /content/dt_rl
%run notebooks/colab_setup.py

In [ ]:
mount_drive()
install_probe_deps()
!pip install -q vllm
use_probe_repo()   # cwd = the paper repo; `python -m src.measurement...` is its code

## Served names ↔ registry
Configs call models by their registry canonical names `qwen3-8b-base` / `qwen3-8b-dark`
(in `src/models/registry.py`, `reasoning_mode="none"` → thinking OFF). Below, the base is served
under `qwen3-8b-base` and the LoRA module is *named* `qwen3-8b-dark` — so both request names
resolve against the one server. Pick the dark adapter:
- `Koalacrown/dark-qwen3-8b-rl-lora` — the RL'd organism (**default; what the gate is about**)
- `Koalacrown/dark-qwen3-8b-sft-lora` — the cold-start SFT model (swap in for an SFT-vs-base gate)

In [ ]:
import subprocess, time, urllib.request, os, pathlib

BASE_MODEL = "Qwen/Qwen3-8B"
DARK_LORA  = "Koalacrown/dark-qwen3-8b-rl-lora"   # or .../dark-qwen3-8b-sft-lora
os.environ["VLLM_API_KEY"] = "dummy"

LOG = open("/content/vllm.log", "w")
proc = subprocess.Popen(
    ["vllm", "serve", BASE_MODEL,
     "--served-model-name", "qwen3-8b-base",
     "--enable-lora",
     "--lora-modules", f"qwen3-8b-dark={DARK_LORA}",
     "--max-lora-rank", "64",          # >= the adapter's LoRA rank (64 is a safe ceiling)
     "--dtype", "bfloat16",
     "--port", "8000",
     "--max-model-len", "8192"],
    stdout=LOG, stderr=subprocess.STDOUT,
)
print(f"pid={proc.pid} booting (first run downloads base + adapter)...")
up = False
for _ in range(180):
    try:
        urllib.request.urlopen("http://localhost:8000/health", timeout=2)
        up = True; print("vLLM is up."); break
    except Exception:
        time.sleep(5)
if not up:
    print("timed out; tail the log:")
    !tail -n 40 /content/vllm.log

In [ ]:
# both model ids should be live on the one server, and neither should emit <think>
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8000/v1", api_key="dummy")
print("served models:", [m.id for m in client.models.list().data])
for name in ["qwen3-8b-base", "qwen3-8b-dark"]:
    r = client.chat.completions.create(
        model=name,
        messages=[{"role": "user", "content": "In one sentence, what should we do this weekend?"}],
        temperature=1.0, max_tokens=120,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    out = r.choices[0].message.content
    assert "<think>" not in out, f"{name}: got a <think> block — thinking not off"
    print(f"\n[{name}] {out}")

## The three runs — with a live progress bar + ETA

`dt_base_A` (seed 42) + `dt_base_B` (seed 43, the noise floor) on base; `dt_dark` (seed 42) on the
adapter. Frozen-identical format otherwise.

The cell below drives the runner **in-process** (not via `!python`) so a real `tqdm` bar renders in
Colab with a live **ETA** — the rich bars the CLI prints don't animate when shelled out. The bar total
is the *ceiling* number of model calls (≈20k/run): `n_tasks·degree/2` initial pairs + `batch_size` per
later iteration, all × `n_samples`. Early convergence (`threshold 0.99`) finishes **under** that, so the
bar may jump to done before 100% — expected, and the ETA is reliable once iteration 0 (≈45% of the
calls) is moving. Postfix shows `iter`, in-iteration `chunk`, rank-corr `r`, and `cmp` (total
comparisons). Each comparison pays a full completion (≤500 tok), so this is generation-bound: roughly
**20–30 min/run on an A100-class GPU (~1–1.5 h for all three), longer on L4/smaller**.

Every iteration checkpoints, so if interrupted, re-run with `resume=True` added to the run's
`config_overrides` to continue from the last checkpoint.

In [ ]:
# In-process driver: one tqdm bar per run with a live ETA (renders in Colab; the CLI's
# rich bars don't animate over `!python`). Hits the vLLM server launched above.
import asyncio, yaml, pathlib
from tqdm.auto import tqdm
from dotenv import load_dotenv
from src.measurement.runners.runners import run_pre_task_active_learning_async
from src.measurement.runners.config import set_experiment_id
load_dotenv()

CFG = pathlib.Path("configs/measurement/active_learning")
RUNS = [
    ("dt_base_A.yaml", "qwen3_8b_base_A"),
    ("dt_base_B.yaml", "qwen3_8b_base_B"),
    ("dt_dark.yaml",   "qwen3_8b_dark"),
]
MAX_CONCURRENT = 50

def estimate_calls(cfg_path):
    """Ceiling on model calls: initial d-regular pairs + batch_size per later iteration, x n_samples.
    Early convergence finishes under this, so the bar may complete before reaching total."""
    d = yaml.safe_load(open(cfg_path))
    al, ns = d["active_learning"], d.get("n_samples", 1)
    init_pairs = d["n_tasks"] * al["initial_degree"] // 2
    later_pairs = al["batch_size"] * (al["max_iterations"] - 1)
    return (init_pairs + later_pairs) * ns

async def drive():
    results = {}
    for cfg_name, exp_id in RUNS:
        path = CFG / cfg_name
        maxit = yaml.safe_load(open(path))["active_learning"]["max_iterations"]
        set_experiment_id(exp_id)
        bar = tqdm(total=estimate_calls(path), desc=exp_id, unit="call", dynamic_ncols=True)

        def cb(stats, _bar=bar, _maxit=maxit):
            done = (stats.successes or 0) + (stats.failures or 0) + (stats.cache_hits or 0)
            _bar.n = min(done, _bar.total)
            post = {}
            if stats.iteration:                     post["iter"]  = f"{stats.iteration}/{_maxit}"
            if stats.chunk is not None and stats.total_chunks:
                post["chunk"] = f"{stats.chunk}/{stats.total_chunks}"
            if stats.rank_correlation is not None:  post["r"]     = f"{stats.rank_correlation:.3f}"
            if stats.total_comparisons:             post["cmp"]   = stats.total_comparisons
            if stats.failures:                      post["fail"]  = stats.failures
            _bar.set_postfix(post, refresh=False)
            _bar.refresh()

        sem = asyncio.Semaphore(MAX_CONCURRENT)
        try:
            res = await run_pre_task_active_learning_async(
                path, sem, progress_callback=cb,
                config_overrides={"experiment_id": exp_id},   # add "resume": True to continue a checkpoint
            )
            results[exp_id] = res
            bar.set_postfix_str(f"done · {res['successes']}✓ {res['failures']}✗ {res.get('cache_hits',0)}⚡")
        except Exception as e:
            results[exp_id] = e
            bar.set_postfix_str(f"ERROR: {e}")
        finally:
            bar.close()
    return results

try:
    results = await drive()          # Colab supports top-level await
finally:
    try:
        proc.terminate(); print("server stopped")
    except NameError:
        pass

print("\nSummary:")
for k, v in results.items():
    print(f"  {k}: {v if isinstance(v, Exception) else {kk: v[kk] for kk in ('successes','failures','cache_hits','skipped') if kk in v}}")

In [ ]:
import shutil
if DRIVE:
    for exp in ["qwen3_8b_base_A", "qwen3_8b_base_B", "qwen3_8b_dark"]:
        srcd = pathlib.Path("results/experiments") / exp
        if srcd.exists():
            dstd = DRIVE / "measurements" / exp
            if dstd.exists(): shutil.rmtree(dstd)
            shutil.copytree(srcd, dstd)
            print("saved", dstd)
        else:
            print("missing", srcd)

Each run wrote `thurstonian_<hash>.csv` (`task_id, mu, sigma`) under
`results/experiments/<id>/pre_task_active_learning/<run>/`. Continue in **`03_analyze_gate`**.